In [2]:
import os
import cv2
import numpy as np
from imutils import paths
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
import copy
import math

# -------------------------------------------------
# Device Configuration
# -------------------------------------------------
print(f"Number of GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}\n')

# -------------------------------------------------
# Hyperparameters (stronger regularization for hard folds)
# -------------------------------------------------
batch_size = 32
num_epochs = 30
learning_rate = 2e-4
weight_decay = 5e-4
img_rows, img_cols = 128, 128
n_folds = 5
early_stop_patience = 7
min_epochs = 6
dropout_p = 0.55
spatial_dropout_p = 0.30

# -------------------------------------------------
# Paths
# -------------------------------------------------
control_folder = r'C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\control'
dyslexia_folder = r'C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\dyslexia'

# -------------------------------------------------
# Load data (group-aware)
# -------------------------------------------------
def load_images_from_folder(folder, label):
    data = []
    for imagePath in paths.list_images(folder):
        image = cv2.imread(imagePath)
        image = cv2.resize(image, (img_cols, img_rows))
        filename = os.path.basename(imagePath)
        group_id = filename.split('_')[0]
        data.append((image, label, group_id))
    return data

control_data = load_images_from_folder(control_folder, 0)
dyslexia_data = load_images_from_folder(dyslexia_folder, 1)
data = control_data + dyslexia_data
np.random.seed(42)
np.random.shuffle(data)

X = np.array([d[0] for d in data], dtype=np.float32)
y = np.array([d[1] for d in data], dtype=np.float32)
groups = np.array([d[2] for d in data])
X /= 255.0

print(f"Total samples: {len(X)} (Control: {(y==0).sum()}, Dyslexia: {(y==1).sum()})")
print(f"Total unique groups/subjects detected: {len(np.unique(groups))}")

# -------------------------------------------------
# Improved CNN Model with BatchNorm
# -------------------------------------------------
class CNNModel(nn.Module):
    def __init__(self, img_rows=128, img_cols=128, dropout_p=0.55, spatial_dropout_p=0.30):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)

        self.dropout_spatial = nn.Dropout2d(spatial_dropout_p)

        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(32)
        self.conv5 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(32)

        self.flat_features = 32 * (img_rows // 4) * (img_cols // 4)

        self.fc1 = nn.Linear(self.flat_features, 64)
        self.bn_fc = nn.BatchNorm1d(64)
        self.dropout_fc = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_spatial(x)

        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = self.pool(x)
        x = self.dropout_spatial(x)

        x = torch.flatten(x, 1)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.dropout_fc(x)
        x = torch.sigmoid(self.fc2(x))
        return x

# -------------------------------------------------
# Stronger on-the-fly augmentation (train only)
# -------------------------------------------------
def augment_batch(x):
    """x: (B, C, H, W) on device, values in [0, 1]"""
    B = x.size(0)

    # Horizontal flip
    if torch.rand(1, device=x.device) < 0.5:
        x = torch.flip(x, dims=[-1])

    # Small random rotation (±12 degrees)
    if torch.rand(1, device=x.device) < 0.5:
        angle = (torch.rand(1, device=x.device).item() * 24 - 12)
        angle_rad = math.radians(angle)
        theta = torch.tensor([
            [math.cos(angle_rad), -math.sin(angle_rad), 0],
            [math.sin(angle_rad),  math.cos(angle_rad), 0]
        ], dtype=torch.float32, device=x.device).unsqueeze(0).repeat(B, 1, 1)
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        x = F.grid_sample(x, grid, align_corners=False, mode='bilinear', padding_mode='border')

    # Brightness / contrast
    scale = 0.70 + 0.60 * torch.rand(1, device=x.device)
    x = (x * scale).clamp(0.0, 1.0)

    # Small brightness shift
    shift = 0.12 * (torch.rand(1, device=x.device) - 0.5)
    x = (x + shift).clamp(0.0, 1.0)

    # Mild Gaussian noise
    if torch.rand(1, device=x.device) < 0.40:
        noise = torch.randn_like(x) * 0.03
        x = (x + noise).clamp(0.0, 1.0)

    return x

# -------------------------------------------------
# Mixup helpers
# -------------------------------------------------
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# -------------------------------------------------
# CutMix helpers
# -------------------------------------------------
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

def cutmix_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]

    # adjust lambda to exactly match pixel ratio
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def cutmix_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# -------------------------------------------------
# Convert data to tensors
# -------------------------------------------------
X_t = torch.tensor(X).permute(0, 3, 1, 2).float()
y_t = torch.tensor(y).float().unsqueeze(1)

# For Out-of-Fold evaluation
oof_true = np.zeros(len(y_t))
oof_prob = np.zeros(len(y_t))

# -------------------------------------------------
# 5-Fold Stratified Group Cross-Validation
# -------------------------------------------------
sgkf = StratifiedGroupKFold(n_splits=n_folds)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups=groups), 1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{n_folds}")
    print(f"{'='*60}")

    X_tr = X_t[train_idx]
    y_tr = y_t[train_idx]
    X_va = X_t[val_idx]
    y_va = y_t[val_idx]

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size, shuffle=False)

    model = CNNModel(img_rows=img_rows, img_cols=img_cols,
                     dropout_p=dropout_p, spatial_dropout_p=spatial_dropout_p).to(device)

    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_fold_val_loss = float('inf')
    best_fold_val_acc  = 0.0
    best_fold_state    = None
    patience_counter   = 0

    for epoch in range(num_epochs):
        # -------------------- Train --------------------
        model.train()
        train_loss = 0.0
        train_correct = 0

        for data_batch, target in train_loader:
            data_batch = data_batch.to(device)
            target = target.to(device)

            # 1. Geometric / intensity augmentation
            data_batch = augment_batch(data_batch)

            # 2. Randomly choose: Normal / Mixup / CutMix
            r = np.random.rand()
            if r < 0.35:          # 35% Mixup
                data_batch, targets_a, targets_b, lam = mixup_data(data_batch, target, alpha=0.2)
                output = model(data_batch)
                loss = mixup_criterion(criterion, output, targets_a, targets_b, lam)
                pred = (output > 0.5).float()
                if lam > 0.5:
                    train_correct += pred.eq(targets_a).sum().item()
                else:
                    train_correct += pred.eq(targets_b).sum().item()

            elif r < 0.70:        # 35% CutMix
                data_batch, targets_a, targets_b, lam = cutmix_data(data_batch, target, alpha=1.0)
                output = model(data_batch)
                loss = cutmix_criterion(criterion, output, targets_a, targets_b, lam)
                pred = (output > 0.5).float()
                if lam > 0.5:
                    train_correct += pred.eq(targets_a).sum().item()
                else:
                    train_correct += pred.eq(targets_b).sum().item()

            else:                 # 30% Normal
                output = model(data_batch)
                loss = criterion(output, target)
                pred = (output > 0.5).float()
                train_correct += pred.eq(target).sum().item()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * data_batch.size(0)

        # -------------------- Validate --------------------
        model.eval()
        val_loss = 0.0
        correct = 0

        with torch.no_grad():
            for data_batch, target in val_loader:
                data_batch = data_batch.to(device)
                target = target.to(device)
                output = model(data_batch)
                val_loss += criterion(output, target).item() * data_batch.size(0)
                pred = (output > 0.5).float()
                correct += pred.eq(target).sum().item()

        train_loss /= len(train_loader.dataset)
        train_acc   = train_correct / len(train_loader.dataset)
        val_loss   /= len(val_loader.dataset)
        accuracy    = correct / len(val_loader.dataset)

        scheduler.step()

        print(f"Epoch {epoch+1:2d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {accuracy:.4f}")

        # -------------------- Early stopping & checkpoint --------------------
        if val_loss < best_fold_val_loss:
            best_fold_val_loss = val_loss
            best_fold_val_acc  = accuracy
            best_fold_state    = copy.deepcopy(model.state_dict())
            patience_counter   = 0
        else:
            if (epoch + 1) >= min_epochs:
                patience_counter += 1
                if patience_counter >= early_stop_patience:
                    print(f" → Early stopping triggered by validation loss at epoch {epoch+1}")
                    break

    # Restore best weights for this fold
    if best_fold_state is not None:
        model.load_state_dict(best_fold_state)

    # ----- Collect Out-of-Fold predictions -----
    model.eval()
    with torch.no_grad():
        val_loader_oof = DataLoader(TensorDataset(X_va, y_va), batch_size=64, shuffle=False)
        probs = []
        for data_batch, _ in val_loader_oof:
            data_batch = data_batch.to(device)
            output = model(data_batch).cpu().numpy().ravel()
            probs.extend(output)

    oof_true[val_idx] = y_va.numpy().ravel()
    oof_prob[val_idx] = np.array(probs)

    fold_results.append({
        "fold": fold,
        "val_loss": best_fold_val_loss,
        "val_acc": best_fold_val_acc
    })

# -------------------------------------------------
# CV Summary
# -------------------------------------------------
print("\n" + "="*60)
print(f"{n_folds}-FOLD CROSS-VALIDATION SUMMARY")
print("="*60)
for res in fold_results:
    print(f"Fold {res['fold']} | Val Loss: {res['val_loss']:.4f} | Val Acc: {res['val_acc']:.4f}")

val_accs   = np.array([r["val_acc"] for r in fold_results])
val_losses = np.array([r["val_loss"] for r in fold_results])

print("-"*60)
print(f"Mean Validation Accuracy: {val_accs.mean():.4f} (+/- {val_accs.std():.4f})")
print(f"Mean Validation Loss: {val_losses.mean():.4f}")
print("="*60)

Number of GPUs available: 2
GPU 0: NVIDIA GeForce RTX 2080 Ti
GPU 1: NVIDIA GeForce RTX 2080 Ti
Using device: cuda:1

Total samples: 7982 (Control: 4566, Dyslexia: 3416)
Total unique groups/subjects detected: 52

FOLD 1/5
Epoch  1/30 | Train Loss: 0.6666 | Train Acc: 0.6038 | Val Loss: 0.5893 | Val Acc: 0.6538
Epoch  2/30 | Train Loss: 0.5003 | Train Acc: 0.7918 | Val Loss: 0.3972 | Val Acc: 0.8361
Epoch  3/30 | Train Loss: 0.3807 | Train Acc: 0.8768 | Val Loss: 0.2821 | Val Acc: 0.8661
Epoch  4/30 | Train Loss: 0.3455 | Train Acc: 0.8924 | Val Loss: 0.2532 | Val Acc: 0.8739
Epoch  5/30 | Train Loss: 0.3203 | Train Acc: 0.9037 | Val Loss: 0.2299 | Val Acc: 0.9125
Epoch  6/30 | Train Loss: 0.3440 | Train Acc: 0.8830 | Val Loss: 0.1842 | Val Acc: 0.9347
Epoch  7/30 | Train Loss: 0.2775 | Train Acc: 0.9245 | Val Loss: 0.1507 | Val Acc: 0.9491
Epoch  8/30 | Train Loss: 0.3065 | Train Acc: 0.9133 | Val Loss: 0.1362 | Val Acc: 0.9569
Epoch  9/30 | Train Loss: 0.2733 | Train Acc: 0.9309 | Val

In [3]:
import numpy as np
import pandas as pd
import re
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# -------------------------------------------------
# 1. Recording-level (Hand-level) evaluation – 52 units
# -------------------------------------------------
recording_df = pd.DataFrame({
    'recording_id': groups,          # e.g. C01Lica, C01Rica, D011Lica...
    'y_true': oof_true.astype(int),
    'y_prob': oof_prob
})

recording_level = recording_df.groupby('recording_id').agg(
    y_true=('y_true', 'first'),
    y_prob_mean=('y_prob', 'mean'),
    n_samples=('y_prob', 'count')
).reset_index()

recording_level['y_pred'] = (recording_level['y_prob_mean'] > 0.5).astype(int)

rec_acc = accuracy_score(recording_level['y_true'], recording_level['y_pred'])

print("="*60)
print("RECORDING-LEVEL PERFORMANCE (52 recordings = 26 subjects × 2 hands)")
print("="*60)
print(f"Number of recordings     : {len(recording_level)}")
print(f"Recording-level Accuracy : {rec_acc:.4f}")
print(classification_report(
    recording_level['y_true'],
    recording_level['y_pred'],
    target_names=['Control', 'Dyslexia']
))
print("Confusion Matrix (Recording-level):")
print(confusion_matrix(recording_level['y_true'], recording_level['y_pred']))

# -------------------------------------------------
# 2. Extract true Participant ID
# -------------------------------------------------
# Examples:
#   C01Lica  → C01
#   C08Rica  → C08
#   D011Lica → D011
recording_level['participant_id'] = recording_level['recording_id'].apply(
    lambda x: re.match(r'([CD]\d+)', x).group(1)
)

print(f"\nNumber of unique participants: {recording_level['participant_id'].nunique()}")
print("Example mapping:")
print(recording_level[['recording_id', 'participant_id']].drop_duplicates().head(8))

# -------------------------------------------------
# 3. True Participant-level evaluation (26 participants)
# -------------------------------------------------
participant_level = recording_level.groupby('participant_id').agg(
    y_true=('y_true', 'first'),
    y_prob_mean=('y_prob_mean', 'mean'),   # average Left + Right hand probabilities
    n_recordings=('recording_id', 'count')
).reset_index()

participant_level['y_pred'] = (participant_level['y_prob_mean'] > 0.5).astype(int)

part_acc = accuracy_score(participant_level['y_true'], participant_level['y_pred'])

print("\n" + "="*60)
print("TRUE PARTICIPANT-LEVEL PERFORMANCE (26 participants)")
print("="*60)
print(f"Number of participants     : {len(participant_level)}")
print(f"Participant-level Accuracy : {part_acc:.4f}")
print()
print(classification_report(
    participant_level['y_true'],
    participant_level['y_pred'],
    target_names=['Control', 'Dyslexia']
))
print("Confusion Matrix (Participant-level):")
print(confusion_matrix(participant_level['y_true'], participant_level['y_pred']))

print("\nRecordings per participant (should be 2 for everyone):")
print(participant_level['n_recordings'].value_counts().sort_index())

RECORDING-LEVEL PERFORMANCE (52 recordings = 26 subjects × 2 hands)
Number of recordings     : 52
Recording-level Accuracy : 0.9808
              precision    recall  f1-score   support

     Control       0.97      1.00      0.98        28
    Dyslexia       1.00      0.96      0.98        24

    accuracy                           0.98        52
   macro avg       0.98      0.98      0.98        52
weighted avg       0.98      0.98      0.98        52

Confusion Matrix (Recording-level):
[[28  0]
 [ 1 23]]

Number of unique participants: 26
Example mapping:
  recording_id participant_id
0     C010Lica           C010
1     C010Rica           C010
2     C011Lica           C011
3     C011Rica           C011
4     C012Lica           C012
5     C012Rica           C012
6     C013Lica           C013
7     C013Rica           C013

TRUE PARTICIPANT-LEVEL PERFORMANCE (26 participants)
Number of participants     : 26
Participant-level Accuracy : 1.0000

              precision    recall  f1-sco

In [4]:
import os
import cv2
import numpy as np
from imutils import paths
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.nn.utils.fusion import fuse_conv_bn_eval
import copy
import math

# -------------------------------------------------
# Device Configuration
# -------------------------------------------------
print(f"Number of GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}\n')

# -------------------------------------------------
# Hyperparameters (stronger regularization for hard folds)
# -------------------------------------------------
batch_size = 32
num_epochs = 30
learning_rate = 2e-4
weight_decay = 5e-4
img_rows, img_cols = 128, 128
n_folds = 5
early_stop_patience = 7
min_epochs = 6
dropout_p = 0.55
spatial_dropout_p = 0.30

# -------------------------------------------------
# Paths
# -------------------------------------------------
control_folder = r'C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\control'
dyslexia_folder = r'C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\dyslexia'

# -------------------------------------------------
# Load data (group-aware)
# -------------------------------------------------
def load_images_from_folder(folder, label):
    data = []
    for imagePath in paths.list_images(folder):
        image = cv2.imread(imagePath)
        image = cv2.resize(image, (img_cols, img_rows))
        filename = os.path.basename(imagePath)
        group_id = filename.split('_')[0]
        data.append((image, label, group_id))
    return data

control_data = load_images_from_folder(control_folder, 0)
dyslexia_data = load_images_from_folder(dyslexia_folder, 1)
data = control_data + dyslexia_data
np.random.seed(42)
np.random.shuffle(data)

X = np.array([d[0] for d in data], dtype=np.float32)
y = np.array([d[1] for d in data], dtype=np.float32)
groups = np.array([d[2] for d in data])
X /= 255.0

print(f"Total samples: {len(X)} (Control: {(y==0).sum()}, Dyslexia: {(y==1).sum()})")
print(f"Total unique groups/subjects detected: {len(np.unique(groups))}")

# -------------------------------------------------
# Full CNN Model with BatchNorm (training architecture)
# -------------------------------------------------
class CNNModel(nn.Module):
    def __init__(self, img_rows=128, img_cols=128, dropout_p=0.55, spatial_dropout_p=0.30):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)

        self.dropout_spatial = nn.Dropout2d(spatial_dropout_p)

        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(32)
        self.conv5 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(32)

        self.flat_features = 32 * (img_rows // 4) * (img_cols // 4)

        self.fc1 = nn.Linear(self.flat_features, 64)
        self.bn_fc = nn.BatchNorm1d(64)
        self.dropout_fc = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_spatial(x)

        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = self.pool(x)
        x = self.dropout_spatial(x)

        x = torch.flatten(x, 1)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.dropout_fc(x)
        x = torch.sigmoid(self.fc2(x))
        return x


# -------------------------------------------------
# Simplified (fused) CNN — mathematically identical outputs
# to CNNModel at inference time, given the same trained weights.
# BatchNorm folded into preceding Conv/Linear; Dropout removed
# (both are already no-ops in eval mode).
# -------------------------------------------------
class CNNModelFused(nn.Module):
    def __init__(self, img_rows=128, img_cols=128):
        super(CNNModelFused, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=5, padding=2)
        self.pool  = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(32, 32, kernel_size=3, padding=1)

        self.flat_features = 32 * (img_rows // 4) * (img_cols // 4)
        self.fc1 = nn.Linear(self.flat_features, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = self.pool(x)

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x


def build_fused_model(trained_model: CNNModel, img_rows=128, img_cols=128) -> CNNModelFused:
    """
    Takes a trained CNNModel (loaded with a fold's best_fold_state)
    and returns a CNNModelFused whose forward() gives numerically
    identical (up to fp32 rounding) outputs, with BN folded and
    Dropout dropped. Only valid for models already in eval() mode
    with frozen BN running stats.
    """
    trained_model.eval()
    fused = CNNModelFused(img_rows=img_rows, img_cols=img_cols)

    fused.conv1 = fuse_conv_bn_eval(trained_model.conv1, trained_model.bn1)
    fused.conv2 = fuse_conv_bn_eval(trained_model.conv2, trained_model.bn2)
    fused.conv3 = fuse_conv_bn_eval(trained_model.conv3, trained_model.bn3)
    fused.conv4 = fuse_conv_bn_eval(trained_model.conv4, trained_model.bn4)
    fused.conv5 = fuse_conv_bn_eval(trained_model.conv5, trained_model.bn5)
    fused.fc1   = fuse_conv_bn_eval(trained_model.fc1, trained_model.bn_fc)

    fused.fc2.load_state_dict(trained_model.fc2.state_dict())

    fused.eval()
    return fused


def verify_fusion_equivalence(model: CNNModel, fused_model: CNNModelFused,
                               x_sample: torch.Tensor, device: torch.device,
                               tol: float = 1e-4) -> float:
    """
    Runs a numerical sanity check confirming the fused model produces
    outputs matching the original (up to fp32 rounding). Returns the
    max absolute difference and prints a pass/fail message.
    """
    model.eval()
    fused_model.eval()
    model = model.to(device)
    fused_model = fused_model.to(device)
    x_sample = x_sample.to(device)

    with torch.no_grad():
        out_orig = model(x_sample)
        out_fused = fused_model(x_sample)
        max_diff = (out_orig - out_fused).abs().max().item()

    status = "PASS" if max_diff < tol else "FAIL"
    print(f"[Fusion check] Max abs diff: {max_diff:.3e}  -> {status} (tol={tol:.0e})")
    return max_diff


# -------------------------------------------------
# Stronger on-the-fly augmentation (train only)
# -------------------------------------------------
def augment_batch(x):
    """x: (B, C, H, W) on device, values in [0, 1]"""
    B = x.size(0)

    # Horizontal flip
    if torch.rand(1, device=x.device) < 0.5:
        x = torch.flip(x, dims=[-1])

    # Small random rotation (±12 degrees)
    if torch.rand(1, device=x.device) < 0.5:
        angle = (torch.rand(1, device=x.device).item() * 24 - 12)
        angle_rad = math.radians(angle)
        theta = torch.tensor([
            [math.cos(angle_rad), -math.sin(angle_rad), 0],
            [math.sin(angle_rad),  math.cos(angle_rad), 0]
        ], dtype=torch.float32, device=x.device).unsqueeze(0).repeat(B, 1, 1)
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        x = F.grid_sample(x, grid, align_corners=False, mode='bilinear', padding_mode='border')

    # Brightness / contrast
    scale = 0.70 + 0.60 * torch.rand(1, device=x.device)
    x = (x * scale).clamp(0.0, 1.0)

    # Small brightness shift
    shift = 0.12 * (torch.rand(1, device=x.device) - 0.5)
    x = (x + shift).clamp(0.0, 1.0)

    # Mild Gaussian noise
    if torch.rand(1, device=x.device) < 0.40:
        noise = torch.randn_like(x) * 0.03
        x = (x + noise).clamp(0.0, 1.0)

    return x

# -------------------------------------------------
# Mixup helpers
# -------------------------------------------------
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# -------------------------------------------------
# CutMix helpers
# -------------------------------------------------
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

def cutmix_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]

    # adjust lambda to exactly match pixel ratio
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def cutmix_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# -------------------------------------------------
# Convert data to tensors
# -------------------------------------------------
X_t = torch.tensor(X).permute(0, 3, 1, 2).float()
y_t = torch.tensor(y).float().unsqueeze(1)

# For Out-of-Fold evaluation
oof_true = np.zeros(len(y_t))
oof_prob = np.zeros(len(y_t))

# Storage for fused models per fold (useful for Grad-CAM/inference reuse later)
fused_models_per_fold = []

# -------------------------------------------------
# 5-Fold Stratified Group Cross-Validation
# -------------------------------------------------
sgkf = StratifiedGroupKFold(n_splits=n_folds)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups=groups), 1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{n_folds}")
    print(f"{'='*60}")

    X_tr = X_t[train_idx]
    y_tr = y_t[train_idx]
    X_va = X_t[val_idx]
    y_va = y_t[val_idx]

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size, shuffle=False)

    model = CNNModel(img_rows=img_rows, img_cols=img_cols,
                     dropout_p=dropout_p, spatial_dropout_p=spatial_dropout_p).to(device)

    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_fold_val_loss = float('inf')
    best_fold_val_acc  = 0.0
    best_fold_state    = None
    patience_counter   = 0

    for epoch in range(num_epochs):
        # -------------------- Train --------------------
        model.train()
        train_loss = 0.0
        train_correct = 0

        for data_batch, target in train_loader:
            data_batch = data_batch.to(device)
            target = target.to(device)

            # 1. Geometric / intensity augmentation
            data_batch = augment_batch(data_batch)

            # 2. Randomly choose: Normal / Mixup / CutMix
            r = np.random.rand()
            if r < 0.35:          # 35% Mixup
                data_batch, targets_a, targets_b, lam = mixup_data(data_batch, target, alpha=0.2)
                output = model(data_batch)
                loss = mixup_criterion(criterion, output, targets_a, targets_b, lam)
                pred = (output > 0.5).float()
                if lam > 0.5:
                    train_correct += pred.eq(targets_a).sum().item()
                else:
                    train_correct += pred.eq(targets_b).sum().item()

            elif r < 0.70:        # 35% CutMix
                data_batch, targets_a, targets_b, lam = cutmix_data(data_batch, target, alpha=1.0)
                output = model(data_batch)
                loss = cutmix_criterion(criterion, output, targets_a, targets_b, lam)
                pred = (output > 0.5).float()
                if lam > 0.5:
                    train_correct += pred.eq(targets_a).sum().item()
                else:
                    train_correct += pred.eq(targets_b).sum().item()

            else:                 # 30% Normal
                output = model(data_batch)
                loss = criterion(output, target)
                pred = (output > 0.5).float()
                train_correct += pred.eq(target).sum().item()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * data_batch.size(0)

        # -------------------- Validate --------------------
        model.eval()
        val_loss = 0.0
        correct = 0

        with torch.no_grad():
            for data_batch, target in val_loader:
                data_batch = data_batch.to(device)
                target = target.to(device)
                output = model(data_batch)
                val_loss += criterion(output, target).item() * data_batch.size(0)
                pred = (output > 0.5).float()
                correct += pred.eq(target).sum().item()

        train_loss /= len(train_loader.dataset)
        train_acc   = train_correct / len(train_loader.dataset)
        val_loss   /= len(val_loader.dataset)
        accuracy    = correct / len(val_loader.dataset)

        scheduler.step()

        print(f"Epoch {epoch+1:2d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {accuracy:.4f}")

        # -------------------- Early stopping & checkpoint --------------------
        if val_loss < best_fold_val_loss:
            best_fold_val_loss = val_loss
            best_fold_val_acc  = accuracy
            best_fold_state    = copy.deepcopy(model.state_dict())
            patience_counter   = 0
        else:
            if (epoch + 1) >= min_epochs:
                patience_counter += 1
                if patience_counter >= early_stop_patience:
                    print(f" → Early stopping triggered by validation loss at epoch {epoch+1}")
                    break

    # Restore best weights for this fold
    if best_fold_state is not None:
        model.load_state_dict(best_fold_state)

    # ----- Build fused (simplified) inference model + verify exact equivalence -----
    model.eval()
    fused_model = build_fused_model(model, img_rows=img_rows, img_cols=img_cols)

    x_check = X_va[:min(8, len(X_va))]
    verify_fusion_equivalence(model, fused_model, x_check, device)

    fused_models_per_fold.append(copy.deepcopy(fused_model).cpu())

    # ----- Collect Out-of-Fold predictions (using fused model; identical outputs, faster) -----
    fused_model = fused_model.to(device)
    fused_model.eval()
    with torch.no_grad():
        val_loader_oof = DataLoader(TensorDataset(X_va, y_va), batch_size=64, shuffle=False)
        probs = []
        for data_batch, _ in val_loader_oof:
            data_batch = data_batch.to(device)
            output = fused_model(data_batch).cpu().numpy().ravel()
            probs.extend(output)

    oof_true[val_idx] = y_va.numpy().ravel()
    oof_prob[val_idx] = np.array(probs)

    fold_results.append({
        "fold": fold,
        "val_loss": best_fold_val_loss,
        "val_acc": best_fold_val_acc
    })

# -------------------------------------------------
# CV Summary
# -------------------------------------------------
print("\n" + "="*60)
print(f"{n_folds}-FOLD CROSS-VALIDATION SUMMARY")
print("="*60)
for res in fold_results:
    print(f"Fold {res['fold']} | Val Loss: {res['val_loss']:.4f} | Val Acc: {res['val_acc']:.4f}")

val_accs   = np.array([r["val_acc"] for r in fold_results])
val_losses = np.array([r["val_loss"] for r in fold_results])

print("-"*60)
print(f"Mean Validation Accuracy: {val_accs.mean():.4f} (+/- {val_accs.std():.4f})")
print(f"Mean Validation Loss: {val_losses.mean():.4f}")
print("="*60)

# -------------------------------------------------
# Note on fused_models_per_fold:
# -------------------------------------------------
# fused_models_per_fold[i] holds the simplified inference-only
# architecture (BN folded, Dropout removed) for fold i+1, verified
# to produce numerically identical outputs to the corresponding
# full CNNModel. Use these for downstream tasks that only need
# forward-pass predictions (e.g. reporting final classifier outputs).
#
# For Grad-CAM++, keep using the original CNNModel objects (with
# best_fold_state loaded) since Grad-CAM++ needs to hook activations
# at a specific conv/BN layer, which no longer exists as a separate
# module once fused into the preceding conv.

Number of GPUs available: 2
GPU 0: NVIDIA GeForce RTX 2080 Ti
GPU 1: NVIDIA GeForce RTX 2080 Ti
Using device: cuda:1

Total samples: 7982 (Control: 4566, Dyslexia: 3416)
Total unique groups/subjects detected: 52

FOLD 1/5
Epoch  1/30 | Train Loss: 0.6720 | Train Acc: 0.5990 | Val Loss: 0.5944 | Val Acc: 0.6871
Epoch  2/30 | Train Loss: 0.4831 | Train Acc: 0.8028 | Val Loss: 0.3790 | Val Acc: 0.8400
Epoch  3/30 | Train Loss: 0.3729 | Train Acc: 0.8806 | Val Loss: 0.3246 | Val Acc: 0.8609
Epoch  4/30 | Train Loss: 0.3317 | Train Acc: 0.9014 | Val Loss: 0.2712 | Val Acc: 0.8687
Epoch  5/30 | Train Loss: 0.3280 | Train Acc: 0.9092 | Val Loss: 0.2199 | Val Acc: 0.9151
Epoch  6/30 | Train Loss: 0.3371 | Train Acc: 0.9006 | Val Loss: 0.2210 | Val Acc: 0.9164
Epoch  7/30 | Train Loss: 0.2706 | Train Acc: 0.9301 | Val Loss: 0.1878 | Val Acc: 0.9334
Epoch  8/30 | Train Loss: 0.3050 | Train Acc: 0.9228 | Val Loss: 0.2125 | Val Acc: 0.9216
Epoch  9/30 | Train Loss: 0.2714 | Train Acc: 0.9369 | Val

In [1]:
import os
import cv2
import numpy as np
from imutils import paths
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
import copy


# ============================================================
# Device Configuration
# ============================================================
print(f"Number of GPUs available: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

device = torch.device(
    'cuda:1' if torch.cuda.is_available() and torch.cuda.device_count() > 1
    else 'cuda:0' if torch.cuda.is_available()
    else 'cpu'
)

print(f"\nUsing device: {device}\n")


# ============================================================
# Hyperparameters
# ============================================================
batch_size = 32
num_epochs = 30

learning_rate = 2e-4
weight_decay = 5e-4

img_rows = 128
img_cols = 128

n_folds = 5

early_stop_patience = 7
min_epochs = 6

dropout_p = 0.55
spatial_dropout_p = 0.30


# ============================================================
# Paths
# ============================================================
control_folder = r'C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\control'
dyslexia_folder = r'C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\dyslexia'


# ============================================================
# Load MI PNG Images
# ============================================================
def load_images_from_folder(folder, label):

    data = []

    image_paths = list(paths.list_images(folder))

    print(f"\nLoading folder:")
    print(folder)
    print(f"Images found: {len(image_paths)}")

    for imagePath in image_paths:

        # ----------------------------------------------------
        # Load PNG as grayscale
        # ----------------------------------------------------
        image = cv2.imread(imagePath, cv2.IMREAD_GRAYSCALE)

        if image is None:
            print(f"WARNING: Could not read {imagePath}")
            continue

        # ----------------------------------------------------
        # Resize to 128 x 128
        # ----------------------------------------------------
        image = cv2.resize(
            image,
            (img_cols, img_rows),
            interpolation=cv2.INTER_AREA
        )

        # ----------------------------------------------------
        # Convert uint8 -> float32
        # ----------------------------------------------------
        image = image.astype(np.float32)

        # ----------------------------------------------------
        # Normalize each MI image to [0, 1]
        #
        # Equivalent to:
        #
        # mi_normalized =
        #     (mi_matrix - mi_matrix.min()) /
        #     (mi_matrix.max() - mi_matrix.min())
        # ----------------------------------------------------
        img_min = image.min()
        img_max = image.max()

        if img_max > img_min:
            image = (image - img_min) / (img_max - img_min)
        else:
            image = np.zeros_like(image, dtype=np.float32)

        # ----------------------------------------------------
        # Extract subject/group ID from filename
        # ----------------------------------------------------
        filename = os.path.basename(imagePath)

        group_id = filename.split('_')[0]

        # ----------------------------------------------------
        # Store
        # image shape currently = H x W
        # ----------------------------------------------------
        data.append(
            (
                image,
                label,
                group_id
            )
        )

    return data


# ============================================================
# Load Control and Dyslexia Data
# ============================================================
control_data = load_images_from_folder(
    control_folder,
    label=0
)

dyslexia_data = load_images_from_folder(
    dyslexia_folder,
    label=1
)

data = control_data + dyslexia_data


# ============================================================
# Shuffle Dataset
# ============================================================
np.random.seed(42)
np.random.shuffle(data)


# ============================================================
# Convert to NumPy
# ============================================================

# Before channel dimension:
# X shape = N x H x W
X = np.array(
    [d[0] for d in data],
    dtype=np.float32
)

y = np.array(
    [d[1] for d in data],
    dtype=np.float32
)

groups = np.array(
    [d[2] for d in data]
)


# ============================================================
# Add Single Channel Dimension
# ============================================================
#
# N x H x W
#      ↓
# N x 1 x H x W
#
# This is equivalent to:
#
# input_tensor =
#     torch.from_numpy(mi_normalized).unsqueeze(0).float()
#
# for an individual MI matrix.
# ============================================================

X = X[:, np.newaxis, :, :]


print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print(
    f"Total samples: {len(X)} "
    f"(Control: {(y == 0).sum()}, "
    f"Dyslexia: {(y == 1).sum()})"
)

print(
    f"Total unique groups/subjects detected: "
    f"{len(np.unique(groups))}"
)

print(f"Input shape: {X.shape}")

print(
    f"Input value range: "
    f"{X.min():.4f} to {X.max():.4f}"
)

print("=" * 60)


# ============================================================
# Improved CNN Model
# ============================================================
class CNNModel(nn.Module):

    def __init__(
        self,
        img_rows=128,
        img_cols=128,
        dropout_p=0.55,
        spatial_dropout_p=0.30
    ):

        super(CNNModel, self).__init__()

        # ----------------------------------------------------
        # First convolution block
        # ----------------------------------------------------
        self.conv1 = nn.Conv2d(
            1,
            16,
            kernel_size=5,
            padding=2
        )

        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(
            16,
            16,
            kernel_size=5,
            padding=2
        )

        self.bn2 = nn.BatchNorm2d(16)

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        self.dropout_spatial = nn.Dropout2d(
            spatial_dropout_p
        )

        # ----------------------------------------------------
        # Second convolution block
        # ----------------------------------------------------
        self.conv3 = nn.Conv2d(
            16,
            32,
            kernel_size=3,
            padding=1
        )

        self.bn3 = nn.BatchNorm2d(32)

        self.conv4 = nn.Conv2d(
            32,
            32,
            kernel_size=3,
            padding=1
        )

        self.bn4 = nn.BatchNorm2d(32)

        self.conv5 = nn.Conv2d(
            32,
            32,
            kernel_size=3,
            padding=1
        )

        self.bn5 = nn.BatchNorm2d(32)

        # ----------------------------------------------------
        # After two 2x2 pooling operations:
        #
        # 128 -> 64 -> 32
        #
        # Therefore:
        # 32 channels x 32 x 32
        # ----------------------------------------------------
        self.flat_features = (
            32 *
            (img_rows // 4) *
            (img_cols // 4)
        )

        # ----------------------------------------------------
        # Fully connected layers
        # ----------------------------------------------------
        self.fc1 = nn.Linear(
            self.flat_features,
            64
        )

        self.bn_fc = nn.BatchNorm1d(64)

        self.dropout_fc = nn.Dropout(
            dropout_p
        )

        self.fc2 = nn.Linear(
            64,
            1
        )

    def forward(self, x):

        # ----------------------------------------------------
        # Block 1
        # ----------------------------------------------------
        x = F.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        x = F.relu(
            self.bn2(
                self.conv2(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_spatial(x)

        # ----------------------------------------------------
        # Block 2
        # ----------------------------------------------------
        x = F.relu(
            self.bn3(
                self.conv3(x)
            )
        )

        x = F.relu(
            self.bn4(
                self.conv4(x)
            )
        )

        x = F.relu(
            self.bn5(
                self.conv5(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_spatial(x)

        # ----------------------------------------------------
        # Flatten
        # ----------------------------------------------------
        x = torch.flatten(
            x,
            start_dim=1
        )

        # ----------------------------------------------------
        # Fully connected
        # ----------------------------------------------------
        x = F.relu(
            self.bn_fc(
                self.fc1(x)
            )
        )

        x = self.dropout_fc(x)

        x = torch.sigmoid(
            self.fc2(x)
        )

        return x


# ============================================================
# Convert NumPy Arrays to PyTorch Tensors
# ============================================================

# X is already:
#
# N x 1 x 128 x 128
#
# Therefore NO permute() is required.

X_t = torch.from_numpy(X).float()

y_t = torch.from_numpy(y).float().unsqueeze(1)


print("\nTensor shapes:")
print(f"X_t: {X_t.shape}")
print(f"y_t: {y_t.shape}")


# ============================================================
# Out-of-Fold Predictions
# ============================================================
oof_true = np.zeros(
    len(y_t),
    dtype=np.float32
)

oof_prob = np.zeros(
    len(y_t),
    dtype=np.float32
)


# ============================================================
# 5-Fold Stratified Group Cross-Validation
# ============================================================
sgkf = StratifiedGroupKFold(
    n_splits=n_folds,
    shuffle=True,
    random_state=42
)

fold_results = []


# ============================================================
# Cross-Validation
# ============================================================
for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(
        X,
        y,
        groups=groups
    ),
    1
):

    print("\n" + "=" * 60)
    print(f"FOLD {fold}/{n_folds}")
    print("=" * 60)

    # --------------------------------------------------------
    # Fold data
    # --------------------------------------------------------
    X_tr = X_t[train_idx]
    y_tr = y_t[train_idx]

    X_va = X_t[val_idx]
    y_va = y_t[val_idx]

    print(
        f"Training samples: {len(train_idx)}"
    )

    print(
        f"Validation samples: {len(val_idx)}"
    )

    print(
        f"Training subjects: "
        f"{len(np.unique(groups[train_idx]))}"
    )

    print(
        f"Validation subjects: "
        f"{len(np.unique(groups[val_idx]))}"
    )

    # --------------------------------------------------------
    # DataLoaders
    # --------------------------------------------------------
    train_loader = DataLoader(
        TensorDataset(
            X_tr,
            y_tr
        ),
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(
            X_va,
            y_va
        ),
        batch_size=batch_size,
        shuffle=False
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------
    model = CNNModel(
        img_rows=img_rows,
        img_cols=img_cols,
        dropout_p=dropout_p,
        spatial_dropout_p=spatial_dropout_p
    ).to(device)

    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------
    criterion = nn.BCELoss()

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------
    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=1e-6
    )

    # --------------------------------------------------------
    # Best model tracking
    # --------------------------------------------------------
    best_fold_val_loss = float('inf')
    best_fold_val_acc = 0.0

    best_fold_state = None

    patience_counter = 0


    # ========================================================
    # Epoch Loop
    # ========================================================
    for epoch in range(num_epochs):

        # ====================================================
        # TRAIN
        # ====================================================
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for data_batch, target in train_loader:

            data_batch = data_batch.to(device)
            target = target.to(device)

            # ------------------------------------------------
            # NO geometric augmentation
            #
            # The MI matrix has channel-specific spatial
            # meaning. Therefore horizontal flipping,
            # rotation and CutMix are not applied.
            # ------------------------------------------------

            # ------------------------------------------------
            # Forward
            # ------------------------------------------------
            output = model(data_batch)

            # ------------------------------------------------
            # Loss
            # ------------------------------------------------
            loss = criterion(
                output,
                target
            )

            # ------------------------------------------------
            # Backpropagation
            # ------------------------------------------------
            optimizer.zero_grad()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            # ------------------------------------------------
            # Statistics
            # ------------------------------------------------
            train_loss += (
                loss.item() *
                data_batch.size(0)
            )

            pred = (
                output > 0.5
            ).float()

            train_correct += (
                pred.eq(target)
                .sum()
                .item()
            )

            train_total += (
                data_batch.size(0)
            )


        # ====================================================
        # VALIDATION
        # ====================================================
        model.eval()

        val_loss = 0.0
        correct = 0
        val_total = 0

        with torch.no_grad():

            for data_batch, target in val_loader:

                data_batch = data_batch.to(device)
                target = target.to(device)

                # ------------------------------------------------
                # Forward
                # ------------------------------------------------
                output = model(data_batch)

                # ------------------------------------------------
                # Validation loss
                # ------------------------------------------------
                loss = criterion(
                    output,
                    target
                )

                val_loss += (
                    loss.item() *
                    data_batch.size(0)
                )

                # ------------------------------------------------
                # Predictions
                # ------------------------------------------------
                pred = (
                    output > 0.5
                ).float()

                correct += (
                    pred.eq(target)
                    .sum()
                    .item()
                )

                val_total += (
                    data_batch.size(0)
                )


        # ====================================================
        # Calculate Metrics
        # ====================================================
        train_loss /= train_total

        train_acc = (
            train_correct /
            train_total
        )

        val_loss /= val_total

        accuracy = (
            correct /
            val_total
        )

        # ----------------------------------------------------
        # Scheduler
        # ----------------------------------------------------
        scheduler.step()


        # ====================================================
        # Print Results
        # ====================================================
        print(
            f"Epoch {epoch + 1:2d}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {accuracy:.4f}"
        )


        # ====================================================
        # Early Stopping
        # ====================================================
        if val_loss < best_fold_val_loss:

            best_fold_val_loss = val_loss
            best_fold_val_acc = accuracy

            best_fold_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            if (epoch + 1) >= min_epochs:

                patience_counter += 1

                if patience_counter >= early_stop_patience:

                    print(
                        f" → Early stopping triggered "
                        f"by validation loss at "
                        f"epoch {epoch + 1}"
                    )

                    break


    # ========================================================
    # Restore Best Model
    # ========================================================
    if best_fold_state is not None:

        model.load_state_dict(
            best_fold_state
        )


    # ========================================================
    # Out-of-Fold Predictions
    # ========================================================
    model.eval()

    probs = []

    with torch.no_grad():

        val_loader_oof = DataLoader(
            TensorDataset(
                X_va,
                y_va
            ),
            batch_size=64,
            shuffle=False
        )

        for data_batch, _ in val_loader_oof:

            data_batch = data_batch.to(device)

            output = (
                model(data_batch)
                .cpu()
                .numpy()
                .ravel()
            )

            probs.extend(output)


    # --------------------------------------------------------
    # Store OOF predictions
    # --------------------------------------------------------
    oof_true[val_idx] = (
        y_va.numpy().ravel()
    )

    oof_prob[val_idx] = np.array(
        probs
    )


    # ========================================================
    # Store Fold Results
    # ========================================================
    fold_results.append(
        {
            "fold": fold,
            "val_loss": best_fold_val_loss,
            "val_acc": best_fold_val_acc
        }
    )


# ============================================================
# CV Summary
# ============================================================
print("\n" + "=" * 60)
print(
    f"{n_folds}-FOLD CROSS-VALIDATION SUMMARY"
)
print("=" * 60)


for res in fold_results:

    print(
        f"Fold {res['fold']} | "
        f"Val Loss: {res['val_loss']:.4f} | "
        f"Val Acc: {res['val_acc']:.4f}"
    )


# ============================================================
# Mean / Standard Deviation
# ============================================================
val_accs = np.array(
    [r["val_acc"] for r in fold_results]
)

val_losses = np.array(
    [r["val_loss"] for r in fold_results]
)


print("-" * 60)

print(
    f"Mean Validation Accuracy: "
    f"{val_accs.mean():.4f} "
    f"(+/- {val_accs.std():.4f})"
)

print(
    f"Mean Validation Loss: "
    f"{val_losses.mean():.4f}"
)

print("=" * 60)

Number of GPUs available: 2
GPU 0: NVIDIA GeForce RTX 2080 Ti
GPU 1: NVIDIA GeForce RTX 2080 Ti

Using device: cuda:1


Loading folder:
C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\control
Images found: 4566

Loading folder:
C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\dyslexia
Images found: 3416

DATASET INFORMATION
Total samples: 7982 (Control: 4566, Dyslexia: 3416)
Total unique groups/subjects detected: 52
Input shape: (7982, 1, 128, 128)
Input value range: 0.0000 to 1.0000

Tensor shapes:
X_t: torch.Size([7982, 1, 128, 128])
y_t: torch.Size([7982, 1])

FOLD 1/5
Training samples: 6451
Validation samples: 1531
Training subjects: 42
Validation subjects: 10
Epoch  1/30 | Train Loss: 0.4448 | Train Acc: 0.7972 | Val Loss: 0.3762 | Val Acc: 0.8570
Epoch  2/30 | Train Loss: 0.1799 | Train Acc: 0.9554 | Val Loss: 0.3064 | Val Acc: 0.8739
Epoch  3/30 | Train Loss: 0.1145 | Train Acc: 0.9727 | Val Loss: 0.2327 | Val Acc: 0.9079
Epoch  4/30 | Train Loss: 0.0702 | Train Acc: 0.9867 | Val Loss: 0.1

In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

import cv2

from imutils import paths

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.decomposition import PCA

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# ============================================================
# 2. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

control_folder = (
    r'C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\control'
)

dyslexia_folder = (
    r'C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\dyslexia'
)


# ------------------------------------------------------------
# Image size
# ------------------------------------------------------------

IMG_SIZE = 128


# ------------------------------------------------------------
# Cross-validation
# ------------------------------------------------------------

N_FOLDS = 5

RANDOM_STATE = 42


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

output_folder = (
    r'C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM'
    r'\flattened_baseline_results'
)

os.makedirs(
    output_folder,
    exist_ok=True
)


# ============================================================
# 3. CHECK IMAGE / DATA ASSUMPTIONS
# ============================================================

print("=" * 70)
print("FLATTENED MI CONNECTIVITY BASELINE")
print("=" * 70)

print("\nIMPORTANT ASSUMPTION:")
print(
    "The PNG files are assumed to represent the numerical MI "
    "matrix directly."
)

print(
    "If the PNGs were created using an RGB colormap such as "
    "'viridis', grayscale conversion cannot recover the "
    "original MI values."
)

print("=" * 70)


# ============================================================
# 4. LOAD MI PNG IMAGES
# ============================================================

def load_mi_images(folder, label):

    data = []

    image_paths = sorted(
        list(paths.list_images(folder))
    )

    print("\nFolder:")
    print(folder)

    print(
        f"Number of images found: "
        f"{len(image_paths)}"
    )

    for image_path in image_paths:

        # ----------------------------------------------------
        # Load as grayscale
        # ----------------------------------------------------

        image = cv2.imread(
            image_path,
            cv2.IMREAD_GRAYSCALE
        )

        if image is None:

            print(
                f"WARNING: Could not read: "
                f"{image_path}"
            )

            continue


        # ----------------------------------------------------
        # Resize if necessary
        # ----------------------------------------------------

        if (
            image.shape[0] != IMG_SIZE
            or
            image.shape[1] != IMG_SIZE
        ):

            image = cv2.resize(
                image,
                (IMG_SIZE, IMG_SIZE),
                interpolation=cv2.INTER_AREA
            )


        # ----------------------------------------------------
        # Convert to float
        # ----------------------------------------------------

        image = image.astype(
            np.float32
        )


        # ----------------------------------------------------
        # Normalize image
        #
        # This corresponds to:
        #
        # (MI - MI_min) /
        # (MI_max - MI_min)
        #
        # but is performed on the PNG intensity values.
        # ----------------------------------------------------

        image_min = image.min()
        image_max = image.max()

        if image_max > image_min:

            image = (
                image - image_min
            ) / (
                image_max - image_min
            )

        else:

            image = np.zeros_like(
                image,
                dtype=np.float32
            )


        # ----------------------------------------------------
        # Extract subject/group ID
        #
        # IMPORTANT:
        #
        # This must correspond to your actual subject ID.
        #
        # Example:
        #
        # subject01_epoch001.png
        #      ↓
        # subject01
        #
        # If your filename structure is different,
        # modify this line.
        # ----------------------------------------------------

        filename = os.path.basename(
            image_path
        )

        group_id = filename.split('_')[0]


        # ----------------------------------------------------
        # Store
        # ----------------------------------------------------

        data.append(
            {
                "matrix": image,
                "label": label,
                "group": group_id,
                "filename": filename,
                "path": image_path
            }
        )

    return data


# ============================================================
# 5. LOAD BOTH CLASSES
# ============================================================

control_data = load_mi_images(
    control_folder,
    label=0
)

dyslexia_data = load_mi_images(
    dyslexia_folder,
    label=1
)


data = (
    control_data +
    dyslexia_data
)


# ============================================================
# 6. SHUFFLE DATA
# ============================================================

rng = np.random.RandomState(
    RANDOM_STATE
)

rng.shuffle(data)


# ============================================================
# 7. CREATE MATRIX / LABEL / GROUP ARRAYS
# ============================================================

mi_matrices = np.array(
    [
        item["matrix"]
        for item in data
    ],
    dtype=np.float32
)

y = np.array(
    [
        item["label"]
        for item in data
    ],
    dtype=np.int64
)

groups = np.array(
    [
        item["group"]
        for item in data
    ]
)

filenames = np.array(
    [
        item["filename"]
        for item in data
    ]
)


# ============================================================
# 8. DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print(
    f"Total samples: "
    f"{len(mi_matrices)}"
)

print(
    f"Control samples: "
    f"{np.sum(y == 0)}"
)

print(
    f"Dyslexia samples: "
    f"{np.sum(y == 1)}"
)

print(
    f"Unique subjects/groups: "
    f"{len(np.unique(groups))}"
)

print(
    f"MI matrix shape: "
    f"{mi_matrices.shape}"
)

print(
    f"MI value range: "
    f"{mi_matrices.min():.6f} "
    f"to "
    f"{mi_matrices.max():.6f}"
)

print("=" * 70)


# ============================================================
# 9. CONVERT MI MATRIX TO FEATURE VECTOR
# ============================================================
#
# 128 channels
#
# Number of unique off-diagonal connections:
#
# C(128, 2)
#
# = 128 * 127 / 2
#
# = 8,128
#
# ============================================================

def matrix_to_feature_vector(
    mi_matrix
):

    if mi_matrix.shape != (
        IMG_SIZE,
        IMG_SIZE
    ):

        raise ValueError(
            f"Expected matrix shape "
            f"{IMG_SIZE} x {IMG_SIZE}, "
            f"but got {mi_matrix.shape}"
        )


    # --------------------------------------------------------
    # Upper triangle
    #
    # k=1 excludes diagonal
    # --------------------------------------------------------

    iu = np.triu_indices(
        IMG_SIZE,
        k=1
    )


    # --------------------------------------------------------
    # Extract unique connections
    # --------------------------------------------------------

    feature_vector = mi_matrix[iu]


    return feature_vector


# ============================================================
# 10. EXTRACT ALL FEATURE VECTORS
# ============================================================

X_features = np.array(
    [
        matrix_to_feature_vector(matrix)
        for matrix in mi_matrices
    ],
    dtype=np.float32
)


# ============================================================
# 11. VERIFY FEATURE DIMENSION
# ============================================================

expected_features = (
    IMG_SIZE *
    (IMG_SIZE - 1)
    // 2
)

print("\n" + "=" * 70)
print("FEATURE EXTRACTION")
print("=" * 70)

print(
    f"MI matrix size: "
    f"{IMG_SIZE} x {IMG_SIZE}"
)

print(
    f"Expected unique connections: "
    f"{expected_features}"
)

print(
    f"Actual feature shape: "
    f"{X_features.shape}"
)

assert X_features.shape[1] == expected_features, (
    "Incorrect number of connectivity features."
)

print(
    "\nFeature extraction verified successfully."
)

print("=" * 70)


# ============================================================
# 12. CREATE EXACT SUBJECT-GROUPED CV SPLITS
# ============================================================
#
# IMPORTANT:
#
# This is the same CV strategy used in your CNN code:
#
# StratifiedGroupKFold
# n_splits=5
# shuffle=True
# random_state=42
#
# Therefore the baseline is evaluated using the same
# subject-grouping principle.
#
# ============================================================

sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)


folds = list(
    sgkf.split(
        X_features,
        y,
        groups=groups
    )
)


# ============================================================
# 13. VERIFY NO SUBJECT LEAKAGE
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING SUBJECT-GROUPED CV")
print("=" * 70)


for fold_number, (
    train_idx,
    val_idx
) in enumerate(
    folds,
    start=1
):

    train_subjects = set(
        groups[train_idx]
    )

    val_subjects = set(
        groups[val_idx]
    )

    overlap = (
        train_subjects &
        val_subjects
    )

    print(
        f"Fold {fold_number}: "
        f"Train subjects = "
        f"{len(train_subjects)}, "
        f"Validation subjects = "
        f"{len(val_subjects)}, "
        f"Overlap = {len(overlap)}"
    )

    if len(overlap) > 0:

        raise RuntimeError(
            f"SUBJECT LEAKAGE DETECTED "
            f"in fold {fold_number}"
        )


print(
    "\nNo subject overlap detected."
)

print("=" * 70)


# ============================================================
# 14. SAVE CV SPLITS
# ============================================================
#
# This is useful for documenting that all models were
# evaluated using the same subject partitions.
#
# ============================================================

split_records = []

for fold_number, (
    train_idx,
    val_idx
) in enumerate(
    folds,
    start=1
):

    for idx in train_idx:

        split_records.append(
            {
                "sample_index": idx,
                "filename": filenames[idx],
                "subject": groups[idx],
                "label": y[idx],
                "fold": fold_number,
                "split": "train"
            }
        )


    for idx in val_idx:

        split_records.append(
            {
                "sample_index": idx,
                "filename": filenames[idx],
                "subject": groups[idx],
                "label": y[idx],
                "fold": fold_number,
                "split": "validation"
            }
        )


split_df = pd.DataFrame(
    split_records
)

split_df.to_csv(
    os.path.join(
        output_folder,
        "subject_grouped_cv_splits.csv"
    ),
    index=False
)


# ============================================================
# 15. MODEL DEFINITIONS
# ============================================================

def create_linear_svm():

    # --------------------------------------------------------
    # PCA is deliberately NOT applied here.
    #
    # Linear SVM can directly handle 8,128 features.
    #
    # StandardScaler is fitted ONLY on the training fold
    # through the Pipeline.
    # --------------------------------------------------------

    model = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),

            (
                "svm",
                LinearSVC(
                    C=1.0,
                    max_iter=10000,
                    class_weight=None,
                    random_state=RANDOM_STATE
                )
            )
        ]
    )

    return model


# ============================================================
# RANDOM FOREST
# ============================================================

def create_random_forest():

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        class_weight=None,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

    return model


# ============================================================
# XGBOOST
# ============================================================

def create_xgboost():

    model = XGBClassifier(

        n_estimators=300,

        max_depth=6,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        objective="binary:logistic",

        eval_metric="logloss",

        random_state=RANDOM_STATE,

        n_jobs=-1,

        tree_method="hist"
    )

    return model


# ============================================================
# 16. MODEL LIST
# ============================================================

models = {

    "Linear SVM":
        create_linear_svm(),

    "Random Forest":
        create_random_forest(),

    "XGBoost":
        create_xgboost()
}


# ============================================================
# 17. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
    y_score
):

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    # --------------------------------------------------------
    # ROC-AUC
    #
    # LinearSVC gives decision_function scores.
    # RF/XGB give probability scores.
    # --------------------------------------------------------

    try:

        auc = roc_auc_score(
            y_true,
            y_score
        )

    except ValueError:

        auc = np.nan


    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


# ============================================================
# 18. STORAGE FOR RESULTS
# ============================================================

all_fold_results = []

oof_results = {}


# ============================================================
# 19. RUN EACH MODEL
# ============================================================

for model_name in models.keys():

    print("\n\n")
    print("#" * 70)
    print(
        f"MODEL: {model_name}"
    )
    print("#" * 70)


    # --------------------------------------------------------
    # Create fresh model
    # --------------------------------------------------------

    if model_name == "Linear SVM":

        model_factory = create_linear_svm

    elif model_name == "Random Forest":

        model_factory = create_random_forest

    elif model_name == "XGBoost":

        model_factory = create_xgboost

    else:

        raise ValueError(
            "Unknown model."
        )


    # --------------------------------------------------------
    # OOF arrays
    # --------------------------------------------------------

    model_oof_pred = np.zeros(
        len(y),
        dtype=np.int64
    )

    model_oof_score = np.zeros(
        len(y),
        dtype=np.float64
    )


    # ========================================================
    # CROSS-VALIDATION
    # ========================================================

    for fold_number, (
        train_idx,
        val_idx
    ) in enumerate(
        folds,
        start=1
    ):

        print("\n" + "-" * 70)

        print(
            f"{model_name} "
            f"— Fold {fold_number}/{N_FOLDS}"
        )

        print("-" * 70)


        # ----------------------------------------------------
        # Training / validation data
        # ----------------------------------------------------

        X_train = X_features[
            train_idx
        ]

        X_val = X_features[
            val_idx
        ]

        y_train = y[
            train_idx
        ]

        y_val = y[
            val_idx
        ]


        print(
            f"Training samples: "
            f"{len(train_idx)}"
        )

        print(
            f"Validation samples: "
            f"{len(val_idx)}"
        )

        print(
            f"Training subjects: "
            f"{len(np.unique(groups[train_idx]))}"
        )

        print(
            f"Validation subjects: "
            f"{len(np.unique(groups[val_idx]))}"
        )


        # ----------------------------------------------------
        # Create a NEW model for every fold
        # ----------------------------------------------------

        model = model_factory()


        # ----------------------------------------------------
        # Fit
        # ----------------------------------------------------

        print(
            "Training model..."
        )

        model.fit(
            X_train,
            y_train
        )


        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        y_pred = model.predict(
            X_val
        )


        # ----------------------------------------------------
        # Scores for ROC-AUC
        # ----------------------------------------------------

        if model_name == "Linear SVM":

            y_score = model.decision_function(
                X_val
            )

        else:

            y_score = model.predict_proba(
                X_val
            )[:, 1]


        # ----------------------------------------------------
        # Calculate metrics
        # ----------------------------------------------------

        metrics = calculate_metrics(
            y_val,
            y_pred,
            y_score
        )


        # ----------------------------------------------------
        # Confusion matrix
        # ----------------------------------------------------

        cm = confusion_matrix(
            y_val,
            y_pred,
            labels=[0, 1]
        )


        tn, fp, fn, tp = (
            cm.ravel()
        )


        # ----------------------------------------------------
        # Store OOF
        # ----------------------------------------------------

        model_oof_pred[
            val_idx
        ] = y_pred

        model_oof_score[
            val_idx
        ] = y_score


        # ----------------------------------------------------
        # Store fold result
        # ----------------------------------------------------

        fold_result = {

            "model": model_name,

            "fold": fold_number,

            "train_samples":
                len(train_idx),

            "validation_samples":
                len(val_idx),

            "train_subjects":
                len(np.unique(
                    groups[train_idx]
                )),

            "validation_subjects":
                len(np.unique(
                    groups[val_idx]
                )),

            "accuracy":
                metrics["accuracy"],

            "precision":
                metrics["precision"],

            "recall":
                metrics["recall"],

            "f1":
                metrics["f1"],

            "auc":
                metrics["auc"],

            "TN":
                tn,

            "FP":
                fp,

            "FN":
                fn,

            "TP":
                tp
        }


        all_fold_results.append(
            fold_result
        )


        # ----------------------------------------------------
        # Print fold metrics
        # ----------------------------------------------------

        print(
            f"\nAccuracy : "
            f"{metrics['accuracy']:.4f}"
        )

        print(
            f"Precision: "
            f"{metrics['precision']:.4f}"
        )

        print(
            f"Recall   : "
            f"{metrics['recall']:.4f}"
        )

        print(
            f"F1       : "
            f"{metrics['f1']:.4f}"
        )

        print(
            f"ROC-AUC  : "
            f"{metrics['auc']:.4f}"
        )

        print(
            "\nConfusion Matrix:"
        )

        print(cm)


    # ========================================================
    # SAVE OOF RESULTS FOR THIS MODEL
    # ========================================================

    oof_results[model_name] = {

        "true": y.copy(),

        "pred": model_oof_pred.copy(),

        "score": model_oof_score.copy()
    }


# ============================================================
# 20. FOLD RESULTS DATAFRAME
# ============================================================

fold_results_df = pd.DataFrame(
    all_fold_results
)


# ============================================================
# 21. SAVE FOLD RESULTS
# ============================================================

fold_results_path = os.path.join(
    output_folder,
    "baseline_fold_results.csv"
)

fold_results_df.to_csv(
    fold_results_path,
    index=False
)


# ============================================================
# 22. CALCULATE MEAN ± SD
# ============================================================

summary_rows = []


for model_name in models.keys():

    model_df = fold_results_df[
        fold_results_df["model"]
        == model_name
    ]


    summary_rows.append(
        {

            "Model":
                model_name,

            "Accuracy_mean":
                model_df["accuracy"].mean(),

            "Accuracy_SD":
                model_df["accuracy"].std(),

            "Precision_mean":
                model_df["precision"].mean(),

            "Precision_SD":
                model_df["precision"].std(),

            "Recall_mean":
                model_df["recall"].mean(),

            "Recall_SD":
                model_df["recall"].std(),

            "F1_mean":
                model_df["f1"].mean(),

            "F1_SD":
                model_df["f1"].std(),

            "AUC_mean":
                model_df["auc"].mean(),

            "AUC_SD":
                model_df["auc"].std()
        }
    )


summary_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# 23. SAVE SUMMARY
# ============================================================

summary_path = os.path.join(
    output_folder,
    "baseline_summary_mean_SD.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 24. OVERALL OOF PERFORMANCE
# ============================================================
#
# This is particularly useful because every sample receives
# exactly one out-of-fold prediction.
#
# ============================================================

oof_rows = []


for model_name in models.keys():

    true_values = (
        oof_results[model_name]["true"]
    )

    predictions = (
        oof_results[model_name]["pred"]
    )

    scores = (
        oof_results[model_name]["score"]
    )


    metrics = calculate_metrics(
        true_values,
        predictions,
        scores
    )


    cm = confusion_matrix(
        true_values,
        predictions,
        labels=[0, 1]
    )


    tn, fp, fn, tp = (
        cm.ravel()
    )


    oof_rows.append(
        {

            "Model":
                model_name,

            "OOF_Accuracy":
                metrics["accuracy"],

            "OOF_Precision":
                metrics["precision"],

            "OOF_Recall":
                metrics["recall"],

            "OOF_F1":
                metrics["f1"],

            "OOF_AUC":
                metrics["auc"],

            "TN":
                tn,

            "FP":
                fp,

            "FN":
                fn,

            "TP":
                tp
        }
    )


oof_summary_df = pd.DataFrame(
    oof_rows
)


# ============================================================
# 25. SAVE OOF SUMMARY
# ============================================================

oof_summary_path = os.path.join(
    output_folder,
    "baseline_OOF_summary.csv"
)

oof_summary_df.to_csv(
    oof_summary_path,
    index=False
)


# ============================================================
# 26. SAVE INDIVIDUAL OOF PREDICTIONS
# ============================================================

oof_prediction_df = pd.DataFrame(
    {
        "filename": filenames,

        "subject": groups,

        "true_label": y,

        "SVM_prediction":
            oof_results[
                "Linear SVM"
            ]["pred"],

        "SVM_score":
            oof_results[
                "Linear SVM"
            ]["score"],

        "RandomForest_prediction":
            oof_results[
                "Random Forest"
            ]["pred"],

        "RandomForest_score":
            oof_results[
                "Random Forest"
            ]["score"],

        "XGBoost_prediction":
            oof_results[
                "XGBoost"
            ]["pred"],

        "XGBoost_score":
            oof_results[
                "XGBoost"
            ]["score"]
    }
)


oof_prediction_path = os.path.join(
    output_folder,
    "baseline_OOF_predictions.csv"
)

oof_prediction_df.to_csv(
    oof_prediction_path,
    index=False
)


# ============================================================
# 27. SAVE CONFUSION MATRICES
# ============================================================

for model_name in models.keys():

    predictions = (
        oof_results[
            model_name
        ]["pred"]
    )


    cm = confusion_matrix(
        y,
        predictions,
        labels=[0, 1]
    )


    safe_name = (
        model_name
        .replace(" ", "_")
    )


    cm_df = pd.DataFrame(
        cm,

        index=[
            "Actual_Control",
            "Actual_Dyslexia"
        ],

        columns=[
            "Predicted_Control",
            "Predicted_Dyslexia"
        ]
    )


    cm_df.to_csv(
        os.path.join(
            output_folder,
            f"{safe_name}_confusion_matrix.csv"
        )
    )


# ============================================================
# 28. PRINT FINAL RESULTS
# ============================================================

print("\n\n")
print("=" * 90)
print("FINAL FLATTENED-CONNECTIVITY BASELINE RESULTS")
print("=" * 90)


print("\n")
print(
    "5-FOLD SUBJECT-GROUPED CROSS-VALIDATION"
)

print(
    "Features per sample: "
    f"{X_features.shape[1]}"
)

print(
    "Number of subjects/groups: "
    f"{len(np.unique(groups))}"
)


print("\n" + "-" * 90)

print(
    "MEAN ± SD ACROSS 5 FOLDS"
)

print("-" * 90)


for _, row in summary_df.iterrows():

    print(
        f"\n{row['Model']}"
    )

    print(
        f"Accuracy : "
        f"{row['Accuracy_mean']:.4f} "
        f"± "
        f"{row['Accuracy_SD']:.4f}"
    )

    print(
        f"Precision: "
        f"{row['Precision_mean']:.4f} "
        f"± "
        f"{row['Precision_SD']:.4f}"
    )

    print(
        f"Recall   : "
        f"{row['Recall_mean']:.4f} "
        f"± "
        f"{row['Recall_SD']:.4f}"
    )

    print(
        f"F1       : "
        f"{row['F1_mean']:.4f} "
        f"± "
        f"{row['F1_SD']:.4f}"
    )

    print(
        f"AUC      : "
        f"{row['AUC_mean']:.4f} "
        f"± "
        f"{row['AUC_SD']:.4f}"
    )


# ============================================================
# 29. OOF RESULTS
# ============================================================

print("\n")
print("-" * 90)

print(
    "OVERALL OUT-OF-FOLD PERFORMANCE"
)

print("-" * 90)


for _, row in oof_summary_df.iterrows():

    print(
        f"\n{row['Model']}"
    )

    print(
        f"Accuracy : "
        f"{row['OOF_Accuracy']:.4f}"
    )

    print(
        f"Precision: "
        f"{row['OOF_Precision']:.4f}"
    )

    print(
        f"Recall   : "
        f"{row['OOF_Recall']:.4f}"
    )

    print(
        f"F1       : "
        f"{row['OOF_F1']:.4f}"
    )

    print(
        f"AUC      : "
        f"{row['OOF_AUC']:.4f}"
    )

    print(
        f"TN = {int(row['TN'])}, "
        f"FP = {int(row['FP'])}, "
        f"FN = {int(row['FN'])}, "
        f"TP = {int(row['TP'])}"
    )


# ============================================================
# 30. OUTPUT FILE LOCATIONS
# ============================================================

print("\n")
print("=" * 90)
print("RESULT FILES")
print("=" * 90)

print(
    "\nFold results:"
)

print(
    fold_results_path
)

print(
    "\nMean ± SD summary:"
)

print(
    summary_path
)

print(
    "\nOverall OOF summary:"
)

print(
    oof_summary_path
)

print(
    "\nIndividual OOF predictions:"
)

print(
    oof_prediction_path
)

print(
    "\nCV split assignments:"
)

print(
    os.path.join(
        output_folder,
        "subject_grouped_cv_splits.csv"
    )
)

print("\n")
print("=" * 90)
print("ANALYSIS COMPLETE")
print("=" * 90)

FLATTENED MI CONNECTIVITY BASELINE

IMPORTANT ASSUMPTION:
The PNG files are assumed to represent the numerical MI matrix directly.
If the PNGs were created using an RGB colormap such as 'viridis', grayscale conversion cannot recover the original MI values.

Folder:
C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\control
Number of images found: 4566

Folder:
C:\Users\RTX2080Ti\Desktop\Dyslexia DL\FCM\dyslexia
Number of images found: 3416

DATASET INFORMATION
Total samples: 7982
Control samples: 4566
Dyslexia samples: 3416
Unique subjects/groups: 52
MI matrix shape: (7982, 128, 128)
MI value range: 0.000000 to 1.000000

FEATURE EXTRACTION
MI matrix size: 128 x 128
Expected unique connections: 8128
Actual feature shape: (7982, 8128)

Feature extraction verified successfully.

VERIFYING SUBJECT-GROUPED CV
Fold 1: Train subjects = 42, Validation subjects = 10, Overlap = 0
Fold 2: Train subjects = 42, Validation subjects = 10, Overlap = 0
Fold 3: Train subjects = 42, Validation subjects = 10, Ove